# Claude로 위키백과 반복 검색하기

[면책: 이 노트북은 Claude 2 모델로 작성되었으며 레거시로 간주됩니다.]

Claude가 곧바로 답할 수 없는 질문도 있습니다. 최신 사건에 관한 것일 수도 있고, Claude가 답을 외우고 있지 않은 매우 세부적인 질문일 수도 있습니다. 걱정하지 마세요! 약간의 프롬프팅과 뼈대만 갖추면 Claude가 웹을 검색해 답을 찾을 수 있습니다. 이 노트북에서는 여러분의 질문에 답하기 위해 위키백과를 검색할 수 있는 가상의 리서치 어시스턴트를 만듭니다. 같은 방식으로 Claude가 더 넓은 웹이나 여러분이 제공한 문서 모음을 검색하게 할 수도 있습니다.

어떤 접근법일까요? 크게 보면 "도구 사용" 범주에 들어갑니다. 검색 도구를 만들고, Claude에 알려 주고, 일하게 두는 것입니다. 의사코드로 표현하면 이렇습니다.

1. 검색 도구의 설명, 가장 잘 쓰는 방법, 그리고 특별한 문자열을 내보내 "호출"하는 방법을 담아 Claude에 프롬프트를 줍니다.
2. Claude에 질문을 던집니다.
3. Claude가 평소처럼 토큰을 생성합니다. 그 특별한 문자열이 나오면 토큰 생성 스트림을 중단하고 검색 API에 질의합니다.
4. 1단계의 프롬프트, 검색 호출 문자열까지 Claude가 생성한 모든 내용, 그리고 API 호출 결과를 합쳐 새 프롬프트를 구성합니다.
5. Claude가 끝났다고 판단할 때까지 반복합니다.

도구 사용과 검색을 위한 프롬프트를 자세히 들여다보겠습니다.

### 프롬프트

In [101]:
# Tool Description Prompt
wikipedia_prompt = """You will be asked a question by a human user. You have access to the following tool to help answer the question. <tool_description> Search Engine Tool * The search engine will exclusively search over Wikipedia for pages similar to your query. It returns for each page its title and full page content. Use this tool if you want to get up-to-date and comprehensive information on a topic to help answer queries. Queries should be as atomic as possible -- they only need to address one part of the user's question. For example, if the user's query is "what is the color of a basketball?", your search query should be "basketball". Here's another example: if the user's question is "Who created the first neural network?", your first query should be "neural network". As you can see, these queries are quite short. Think keywords, not phrases. * At any time, you can make a call to the search engine using the following syntax: <search_query>query_word</search_query>. * You'll then get results back in <search_result> tags.</tool_description>"""
print(wikipedia_prompt)

You will be asked a question by a human user. You have access to the following tool to help answer the question. <tool_description> Search Engine Tool * The search engine will exclusively search over Wikipedia for pages similar to your query. It returns for each page its title and full page content. Use this tool if you want to get up-to-date and comprehensive information on a topic to help answer queries. Queries should be as atomic as possible -- they only need to address one part of the user's question. For example, if the user's query is "what is the color of a basketball?", your search query should be "basketball". Here's another example: if the user's question is "Who created the first neural network?", your first query should be "neural network". As you can see, these queries are quite short. Think keywords, not phrases. * At any time, you can make a call to the search engine using the following syntax: <search_query>query_word</search_query>. * You'll then get results back in <

이 프롬프트에는 위키백과를 제대로 검색하는 방법에 대한 조언이 많이 담겨 있습니다. 우리는 구글에 아무 말이나 입력해도 질의 해석 로직이 워낙 좋아서 괜찮은 결과를 얻는 데 익숙합니다. 위키백과 검색은 그렇지 않습니다. 예를 들어 "What's the best way to purchase potatoes in the United Arab Emirates"라는 질의를 생각해 보세요. [위키백과에서 이 질의의 상위 결과](https://en.wikipedia.org/w/index.php?search=What%27s+the+best+way+to+purchase+potatoes+in+the+United+Arab+Emirates&title=Special:Search&profile=advanced&fulltext=1&ns0=1)는 미국의 노예제, 1973년 석유 파동, Wendy's, Tim Horton's(??)입니다. 반면 구글은 곧바로 Carrefour UAE로 안내합니다.

또 다른 차이는 위키백과 검색이 문서 전체를 반환한다는 점입니다. 벡터 검색에서는 더 좁은 청크를 얻을 수 있으므로, 결과 개수를 늘리거나 더 구체적인 질의를 쓰거나, 둘 다 해야 할 수도 있습니다. 큰 그림에서의 교훈은 이런 선택에 따라 결과가 크게 달라질 수 있으니 주의하라는 것입니다!

In [103]:
retrieval_prompt = """Before beginning to research the user's question, first think for a moment inside <scratchpad> tags about what information is necessary for a well-informed answer. If the user's question is complex, you may need to decompose the query into multiple subqueries and execute them individually. Sometimes the search engine will return empty search results, or the search results may not contain the information you need. In such cases, feel free to try again with a different query.

After each call to the Search Engine Tool, reflect briefly inside <search_quality></search_quality> tags about whether you now have enough information to answer, or whether more information is needed. If you have all the relevant information, write it in <information></information> tags, WITHOUT actually answering the question. Otherwise, issue a new search.

Here is the user's question: <question>{query}</question> Remind yourself to make short queries in your scratchpad as you plan out your strategy."""
print(retrieval_prompt)

Before beginning to research the user's question, first think for a moment inside <scratchpad> tags about what information is necessary for a well-informed answer. If the user's question is complex, you may need to decompose the query into multiple subqueries and execute them individually. Sometimes the search engine will return empty search results, or the search results may not contain the information you need. In such cases, feel free to try again with a different query. 

After each call to the Search Engine Tool, reflect briefly inside <search_quality></search_quality> tags about whether you now have enough information to answer, or whether more information is needed. If you have all the relevant information, write it in <information></information> tags, WITHOUT actually answering the question. Otherwise, issue a new search.

Here is the user's question: <question>{query}</question> Remind yourself to make short queries in your scratchpad as you plan out your strategy.



여기서 스크래치패드를 쓰는 이유는 일반적인 생각의 사슬과 같습니다. Claude가 질문에 답할 일관된 계획을 세우게 해 줍니다. 검색 품질 성찰은 Claude가 끈기 있게 파고들어, 관련 정보를 다 모으기도 전에 성급히 답해 버리지 않도록 유도하기 위한 것입니다. 그런데 왜 Claude에게 정보를 종합하라고 하고 바로 답하지 말라고 할까요?

In [104]:
answer_prompt = "Here is a user query: <query>{query}</query>. Here is some relevant information: <information>{information}</information>. Please answer the question using the relevant information."
print(answer_prompt)

Here is a user query: <query>{query}</query>. Here is some relevant information: <information>{information}</information>. Please answer the question using the relevant information.


정보를 추출해 새 질의로 Claude에 제시하면, Claude가 그 정보를 올바른 답으로 종합하는 데 온전히 집중할 수 있습니다. 이 단계가 없으면 Claude가 먼저 답을 정해 놓고 검색 결과로 그것을 "정당화"하는 경우가 있었습니다. 결과가 답을 이끄는 것이 아니라요.

이제 검색 + 검색 결과 수집 + 재프롬프팅의 의사코드를 구현하는 코드가 이어집니다.

### 검색 구현

In [88]:
import re
from abc import abstractmethod
from dataclasses import dataclass

import wikipedia
from anthropic import AI_PROMPT, HUMAN_PROMPT, Anthropic


@dataclass
class SearchResult:
    """
    A single search result.
    """

    content: str


class SearchTool:
    """
    A search tool that can run a query and return a formatted string of search results.
    """

    def __init__():
        pass

    @abstractmethod
    def raw_search(self, query: str, n_search_results_to_use: int) -> list[SearchResult]:
        """
        Runs a query using the searcher, then returns the raw search results without formatting.

        :param query: The query to run.
        :param n_search_results_to_use: The number of results to return.
        """
        raise NotImplementedError()

    @abstractmethod
    def process_raw_search_results(
        self,
        results: list[SearchResult],
    ) -> list[str]:
        """
        Extracts the raw search content from the search results and returns a list of strings that can be passed to Claude.

        :param results: The search results to extract.
        """
        raise NotImplementedError()

    def search_results_to_string(self, extracted: list[str]) -> str:
        """
        Joins and formats the extracted search results as a string.

        :param extracted: The extracted search results to format.
        """
        result = "\n".join(
            [
                f'<item index="{i + 1}">\n<page_content>\n{r}\n</page_content>\n</item>'
                for i, r in enumerate(extracted)
            ]
        )
        return result

    def wrap_search_results(self, extracted: list[str]) -> str:
        """
        Formats the extracted search results as a string, including the <search_results> tags.

        :param extracted: The extracted search results to format.
        """
        return f"\n<search_results>\n{self.search_results_to_string(extracted)}\n</search_results>"

    def search(self, query: str, n_search_results_to_use: int) -> str:
        raw_search_results = self.raw_search(query, n_search_results_to_use)
        processed_search_results = self.process_raw_search_results(raw_search_results)
        displayable_search_results = self.wrap_search_results(processed_search_results)
        return displayable_search_results

In [ ]:
@dataclass
class WikipediaSearchResult(SearchResult):
    title: str


class WikipediaSearchTool(SearchTool):
    def __init__(self, truncate_to_n_tokens: int | None = 5000):
        self.truncate_to_n_tokens = truncate_to_n_tokens
        if truncate_to_n_tokens is not None:
            self.tokenizer = Anthropic().get_tokenizer()

    def raw_search(self, query: str, n_search_results_to_use: int) -> list[WikipediaSearchResult]:
        search_results = self._search(query, n_search_results_to_use)
        return search_results

    def process_raw_search_results(self, results: list[WikipediaSearchResult]) -> list[str]:
        processed_search_results = [
            f"Page Title: {result.title.strip()}\nPage Content:\n{self.truncate_page_content(result.content)}"
            for result in results
        ]
        return processed_search_results

    def truncate_page_content(self, page_content: str) -> str:
        if self.truncate_to_n_tokens is None:
            return page_content.strip()
        else:
            return self.tokenizer.decode(
                self.tokenizer.encode(page_content).ids[: self.truncate_to_n_tokens]
            ).strip()

    def _search(self, query: str, n_search_results_to_use: int) -> list[WikipediaSearchResult]:
        results: list[str] = wikipedia.search(query)
        search_results: list[WikipediaSearchResult] = []
        for result in results:
            if len(search_results) >= n_search_results_to_use:
                break
            try:
                page = wikipedia.page(result)
                print(page.url)
            except wikipedia.exceptions.WikipediaException:
                # The Wikipedia API is a little flaky, so we just skip over pages that fail to load
                continue
            content = page.content
            title = page.title
            search_results.append(WikipediaSearchResult(content=content, title=title))
        return search_results

In [100]:
def extract_between_tags(tag: str, string: str, strip: bool = True) -> list[str]:
    ext_list = re.findall(rf"<{tag}\s?>(.+?)</{tag}\s?>", string, re.DOTALL)
    if strip:
        ext_list = [e.strip() for e in ext_list]
    return ext_list


class ClientWithRetrieval(Anthropic):
    def __init__(self, search_tool: SearchTool, verbose: bool = True, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.search_tool = search_tool
        self.verbose = verbose

    # Helper methods
    def _search_query_stop(
        self, partial_completion: str, n_search_results_to_use: int
    ) -> tuple[list[SearchResult], str]:
        search_query = extract_between_tags("search_query", partial_completion + "</search_query>")
        if search_query is None:
            raise Exception(
                "Completion with retrieval failed as partial completion returned mismatched <search_query> tags."
            )
        print(f"Running search query against SearchTool: {search_query}")
        search_results = self.search_tool.raw_search(search_query, n_search_results_to_use)
        extracted_search_results = self.search_tool.process_raw_search_results(search_results)
        formatted_search_results = self.search_tool.wrap_search_results(extracted_search_results)
        return search_results, formatted_search_results

    def retrieve(
        self,
        query: str,
        model: str,
        n_search_results_to_use: int = 3,
        stop_sequences: list[str] = None,
        max_tokens_to_sample: int = 1000,
        max_searches_to_try: int = 5,
        temperature: float = 1.0,
    ) -> tuple[list[SearchResult], str]:
        if stop_sequences is None:
            stop_sequences = [HUMAN_PROMPT]
        prompt = (
            f"{HUMAN_PROMPT} {wikipedia_prompt} {retrieval_prompt.format(query=query)}{AI_PROMPT}"
        )
        starting_prompt = prompt
        print("Starting prompt:", starting_prompt)
        token_budget = max_tokens_to_sample
        all_raw_search_results: list[SearchResult] = []
        for tries in range(max_searches_to_try):
            partial_completion = self.completions.create(
                prompt=prompt,
                stop_sequences=stop_sequences + ["</search_query>"],
                model=model,
                max_tokens_to_sample=token_budget,
                temperature=temperature,
            )
            partial_completion, stop_reason, stop_seq = (
                partial_completion.completion,
                partial_completion.stop_reason,
                partial_completion.stop,
            )
            print(partial_completion)
            token_budget -= self.count_tokens(partial_completion)
            prompt += partial_completion
            if stop_reason == "stop_sequence" and stop_seq == "</search_query>":
                print(f"Attempting search number {tries}.")
                raw_search_results, formatted_search_results = self._search_query_stop(
                    partial_completion, n_search_results_to_use
                )
                prompt += "</search_query>" + formatted_search_results
                all_raw_search_results += raw_search_results
            else:
                break
        final_model_response = prompt[len(starting_prompt) :]
        return all_raw_search_results, final_model_response

    # Main methods
    def completion_with_retrieval(
        self,
        query: str,
        model: str,
        n_search_results_to_use: int = 3,
        stop_sequences: list[str] = None,
        max_tokens_to_sample: int = 1000,
        max_searches_to_try: int = 5,
        temperature: float = 1.0,
    ) -> str:
        if stop_sequences is None:
            stop_sequences = [HUMAN_PROMPT]
        _, retrieval_response = self.retrieve(
            query,
            model=model,
            n_search_results_to_use=n_search_results_to_use,
            stop_sequences=stop_sequences,
            max_tokens_to_sample=max_tokens_to_sample,
            max_searches_to_try=max_searches_to_try,
            temperature=temperature,
        )
        information = extract_between_tags("information", retrieval_response)[-1]
        prompt = f"{HUMAN_PROMPT} {answer_prompt.format(query=query, information=information)}{AI_PROMPT}"
        print("Summarizing:\n", prompt)
        answer = self.completions.create(
            prompt=prompt, model=model, temperature=temperature, max_tokens_to_sample=1000
        ).completion
        return answer

### 질의 실행하기

질의를 실행할 준비가 되었습니다! 다음 조건에 맞는 것을 골라 보겠습니다.
- 최근의 것이어서 Claude의 학습 데이터에 있을 가능성이 낮고,
- 복합적이어서 여러 번의 검색이 필요한 것.

In [98]:
import os

# Create a searcher
wikipedia_search_tool = WikipediaSearchTool()
ANTHROPIC_SEARCH_MODEL = "claude-2"

client = ClientWithRetrieval(
    api_key=os.environ["ANTHROPIC_API_KEY"], verbose=True, search_tool=wikipedia_search_tool
)

query = "Which movie came out first: Oppenheimer, or Are You There God It's Me Margaret?"

augmented_response = client.completion_with_retrieval(
    query=query,
    model=ANTHROPIC_SEARCH_MODEL,
    n_search_results_to_use=1,
    max_searches_to_try=5,
    max_tokens_to_sample=1000,
    temperature=0,
)
print(augmented_response)

Starting prompt: 

Human: You will be asked a question by a human user. You have access to the following tool to help answer the question. <tool_description> Search Engine Tool * The search engine will exclusively search over Wikipedia for pages similar to your query. It returns for each page its title and full page content. Use this tool if you want to get up-to-date and comprehensive information on a topic to help answer queries. Queries should be as atomic as possible -- they only need to address one part of the user's question. For example, if the user's query is "what is the color of a basketball?", your search query should be "basketball". Here's another example: if the user's question is "Who created the first neural network?", your first query should be "neural network". As you can see, these queries are quite short. Think keywords, not phrases. * At any time, you can make a call to the search engine using the following syntax: <search_query>query_word</search_query>. * You'll 

좋습니다. Claude가 계획을 세우고, 질의를 실행하고, 정보를 종합해 정확한 답을 내놓았습니다. 참고: 추가 정보 추출 단계가 없으면 Claude가 영화 개봉일은 정확히 알아내고도 최종 답변에서 순서를 틀리는 경우가 있었습니다. 하나 더 해 보겠습니다.

In [99]:
augmented_response = client.completion_with_retrieval(
    query="Who won the 2023 NBA championship? Who was that team's best player in the year 2009?",
    model=ANTHROPIC_SEARCH_MODEL,
    n_search_results_to_use=1,
    max_searches_to_try=5,
    max_tokens_to_sample=1000,
    temperature=0,
)
print(augmented_response)

Starting prompt: 

Human: You will be asked a question by a human user. You have access to the following tool to help answer the question. <tool_description> Search Engine Tool * The search engine will exclusively search over Wikipedia for pages similar to your query. It returns for each page its title and full page content. Use this tool if you want to get up-to-date and comprehensive information on a topic to help answer queries. Queries should be as atomic as possible -- they only need to address one part of the user's question. For example, if the user's query is "what is the color of a basketball?", your search query should be "basketball". Here's another example: if the user's question is "Who created the first neural network?", your first query should be "neural network". As you can see, these queries are quite short. Think keywords, not phrases. * At any time, you can make a call to the search engine using the following syntax: <search_query>query_word</search_query>. * You'll 

자, 이렇게 됩니다! 검색 도구 코드가 깔끔하게 추상화되어 있어서 약간만 수정하면 원하는 검색 API를 쓰도록 바꿀 수 있다는 점을 눈치채셨을 겁니다. 다만 Claude가 그 도구를 잘 쓰는 데 필요한 요령은 반드시 설명해 주세요. 성능을 더 끌어올리려면 이상적인 질의 계획과 질의 구조에 대한 퓨샷 예시를 Claude에 제공할 수도 있습니다.